In [66]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ishaans23/cbis-ddsm-training/cache_info.json
/kaggle/input/datasets/ishaans23/cbis-ddsm-training/excluded_mass_cases.csv
/kaggle/input/datasets/ishaans23/cbis-ddsm-training/image_metadata.csv
/kaggle/input/datasets/ishaans23/cbis-ddsm-training/shared_mass_calc_audit.csv
/kaggle/input/datasets/ishaans23/cbis-ddsm-training/masks/P_00001_LEFT_CC.png
/kaggle/input/datasets/ishaans23/cbis-ddsm-training/masks/P_00001_LEFT_MLO.png
/kaggle/input/datasets/ishaans23/cbis-ddsm-training/masks/P_00004_LEFT_CC.png
/kaggle/input/datasets/ishaans23/cbis-ddsm-training/masks/P_00004_LEFT_MLO.png
/kaggle/input/datasets/ishaans23/cbis-ddsm-training/masks/P_00004_RIGHT_MLO.png
/kaggle/input/datasets/ishaans23/cbis-ddsm-training/masks/P_00005_RIGHT_CC.png
/kaggle/input/datasets/ishaans23/cbis-ddsm-training/masks/P_00005_RIGHT_MLO.png
/kaggle/input/datasets/ishaans23/cbis-ddsm-training/masks/P_00007_LEFT_CC.png
/kaggle/input/datasets/ishaans23/cbis-ddsm-training/masks/P_00007_LEFT_MLO.

In [67]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.transforms.functional import normalize

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

from tqdm.auto import tqdm

In [68]:
SEED = 42

TARGET_HEIGHT = 1024
TARGET_WIDTH = 640

BATCH_SIZE = 4

# Keep this at 0 initially because Kaggle previously produced
# multiprocessing DataLoader cleanup errors.
NUM_WORKERS = 0

HEAD_EPOCHS = 2

FINETUNE_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 2

HEAD_LR = 1e-3

LAYER4_LR = 1e-5
FC_FINETUNE_LR = 1e-4

WEIGHT_DECAY = 1e-4


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch: {torch.__version__}")

PyTorch: 2.11.0+cu128


In [69]:
KAGGLE_INPUT_ROOT = Path("/kaggle/input")

candidate_roots = []

for cache_info_path in KAGGLE_INPUT_ROOT.rglob("cache_info.json"):
    candidate_root = cache_info_path.parent

    required_paths = [
        candidate_root / "npy",
        candidate_root / "masks",
        candidate_root / "splits",
    ]

    if all(path.exists() for path in required_paths):
        candidate_roots.append(candidate_root)


assert len(candidate_roots) > 0, (
    "Could not find the cached CBIS-DDSM dataset under /kaggle/input."
)

# Prefer the expected cache folder name if multiple candidates exist.
candidate_roots = sorted(
    candidate_roots,
    key=lambda path: (
        "cbis_ddsm_model_ready_1024x640" not in path.name.lower(),
        len(str(path)),
    )
)

PROJECT_ROOT = candidate_roots[0]

print("Using dataset:")
print(PROJECT_ROOT)

with open(PROJECT_ROOT / "cache_info.json", "r") as f:
    cache_info = json.load(f)

print("\nCache information:")
print(json.dumps(cache_info, indent=2))

Using dataset:
/kaggle/input/datasets/ishaans23/cbis-ddsm-training

Cache information:
{
  "target_height": 1024,
  "target_width": 640,
  "image_dtype": "uint16",
  "mask_dtype": "uint8",
  "mask_values": [
    0,
    255
  ],
  "image_interpolation": "cv2.INTER_AREA",
  "mask_interpolation": "cv2.INTER_NEAREST",
  "alignment": {
    "horizontal": "left",
    "vertical": "center"
  },
  "num_images": 2687,
  "source_dataset": "/kaggle/input/datasets/ishaans23/cbis-ddssm/kaggle_upload_corrected"
}


In [70]:
SPLITS_DIR = PROJECT_ROOT / "splits"

train_df = pd.read_csv(SPLITS_DIR / "train.csv")
val_df = pd.read_csv(SPLITS_DIR / "val.csv")
test_df = pd.read_csv(SPLITS_DIR / "test.csv")

print(f"Train: {len(train_df)}")
print(f"Val:   {len(val_df)}")
print(f"Test:  {len(test_df)}")
print(f"Total: {len(train_df) + len(val_df) + len(test_df)}")

Train: 1885
Val:   408
Test:  394
Total: 2687


In [71]:
def validate_dataset_splits(
    train_df,
    val_df,
    test_df,
    root_dir,
):
    splits = {
        "train": train_df,
        "val": val_df,
        "test": test_df,
    }

    for split_name, df in splits.items():

        assert df["malignant"].isin([0, 1]).all(), (
            f"{split_name}: invalid binary labels found"
        )

        missing_images = [
            relative_path
            for relative_path in df["npy_path"]
            if not (root_dir / relative_path).exists()
        ]

        assert len(missing_images) == 0, (
            f"{split_name}: "
            f"{len(missing_images)} cached images missing"
        )

    train_patients = set(train_df["patient_id"])
    val_patients = set(val_df["patient_id"])
    test_patients = set(test_df["patient_id"])

    assert train_patients.isdisjoint(val_patients), (
        "Patient leakage between train and validation"
    )

    assert train_patients.isdisjoint(test_patients), (
        "Patient leakage between train and test"
    )

    assert val_patients.isdisjoint(test_patients), (
        "Patient leakage between validation and test"
    )

    all_keys = pd.concat(
        [
            train_df["image_key"],
            val_df["image_key"],
            test_df["image_key"],
        ]
    )

    assert not all_keys.duplicated().any(), (
        "Duplicate image_key found across splits"
    )

    print("Dataset validation passed.")
    print(
        f"{len(train_df)} train | "
        f"{len(val_df)} validation | "
        f"{len(test_df)} test"
    )


validate_dataset_splits(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    root_dir=PROJECT_ROOT,
)

Dataset validation passed.
1885 train | 408 validation | 394 test


In [72]:
sample_row = train_df.iloc[0]

sample_path = PROJECT_ROOT / sample_row["npy_path"]
sample_image = np.load(sample_path)

print("Image key:", sample_row["image_key"])
print("Shape:", sample_image.shape)
print("dtype:", sample_image.dtype)
print("Min:", sample_image.min())
print("Max:", sample_image.max())

assert sample_image.shape == (
    TARGET_HEIGHT,
    TARGET_WIDTH,
)

assert sample_image.dtype == np.uint16

print("\nCached image format verified.")

Image key: P_00001_LEFT_CC
Shape: (1024, 640)
dtype: uint16
Min: 0
Max: 65535

Cached image format verified.


In [73]:
weights = ResNet50_Weights.DEFAULT

weight_transforms = weights.transforms()

IMAGENET_MEAN = weight_transforms.mean
IMAGENET_STD = weight_transforms.std

print("Weights:", weights)
print("Mean:", IMAGENET_MEAN)
print("Std:", IMAGENET_STD)

Weights: ResNet50_Weights.IMAGENET1K_V2
Mean: [0.485, 0.456, 0.406]
Std: [0.229, 0.224, 0.225]


In [74]:
class CBISClassificationDataset(Dataset):

    def __init__(
        self,
        dataframe,
        root_dir,
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.root_dir = Path(root_dir)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        image_path = (
            self.root_dir / row["npy_path"]
        )

        image = np.load(
            image_path,
            mmap_mode="r",
        )

        if image.shape != (
            TARGET_HEIGHT,
            TARGET_WIDTH,
        ):
            raise ValueError(
                f"{row['image_key']}: "
                f"expected "
                f"{TARGET_HEIGHT}x{TARGET_WIDTH}, "
                f"got {image.shape}"
            )

        # uint16 -> float32 [0, 1]
        image = np.asarray(
            image,
            dtype=np.float32,
        ).copy()

        image /= 65535.0

        # [H, W] -> [1, H, W]
        image = torch.from_numpy(image).unsqueeze(0)

        # Grayscale -> 3-channel RGB-compatible tensor
        image = image.repeat(3, 1, 1)

        image = normalize(
            image,
            mean=IMAGENET_MEAN,
            std=IMAGENET_STD,
        )

        label = int(row["malignant"])

        image_key = row["image_key"]

        return image, label, image_key

In [75]:
train_dataset = CBISClassificationDataset(
    train_df,
    PROJECT_ROOT,
)

val_dataset = CBISClassificationDataset(
    val_df,
    PROJECT_ROOT,
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

Train samples: 1885
Validation samples: 408


In [76]:
PIN_MEMORY = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Train batches: 472
Validation batches: 102


In [77]:
images, labels, image_keys = next(
    iter(train_loader)
)

print("Images:", images.shape)
print("Labels:", labels.shape)

print("Image dtype:", images.dtype)
print("Label dtype:", labels.dtype)

print("Example keys:")
print(image_keys[:2])

Images: torch.Size([4, 3, 1024, 640])
Labels: torch.Size([4])
Image dtype: torch.float32
Label dtype: torch.int64
Example keys:
('P_01292_RIGHT_CC', 'P_01349_LEFT_MLO')


In [78]:
device = torch.device(
    "cuda:0"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():

    print(
        "GPUs available:",
        torch.cuda.device_count(),
    )

    for i in range(
        torch.cuda.device_count()
    ):
        print(
            f"GPU {i}: "
            f"{torch.cuda.get_device_name(i)}"
        )

Device: cuda:0
GPUs available: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [79]:
def build_classifier(weights):

    model = resnet50(
        weights=weights
    )

    in_features = (
        model.fc.in_features
    )

    model.fc = nn.Linear(
        in_features,
        1,
    )

    # Freeze entire network
    for param in model.parameters():
        param.requires_grad = False

    # Train classifier head
    for param in model.fc.parameters():
        param.requires_grad = True

    return model


model = build_classifier(weights)
model = model.to(device)

print(model.fc)

Linear(in_features=2048, out_features=1, bias=True)


In [80]:
def summarize_trainable_parameters(model):

    total_params = sum(
        param.numel()
        for param in model.parameters()
    )

    trainable_params = sum(
        param.numel()
        for param in model.parameters()
        if param.requires_grad
    )

    print(
        f"Total parameters: "
        f"{total_params:,}"
    )

    print(
        f"Trainable parameters: "
        f"{trainable_params:,}"
    )

    print("\nTrainable layers:")

    for name, param in model.named_parameters():
        if param.requires_grad:
            print(f"  {name}")


summarize_trainable_parameters(model)

Total parameters: 23,510,081
Trainable parameters: 2,049

Trainable layers:
  fc.weight
  fc.bias


In [81]:
OUTPUT_DIR = Path(
    "/kaggle/working/classifier_experiment"
)

CHECKPOINT_DIR = (
    OUTPUT_DIR / "checkpoints"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

HEAD_BEST_PATH = (
    CHECKPOINT_DIR
    / "resnet50_head_best.pth"
)

HEAD_LATEST_PATH = (
    CHECKPOINT_DIR
    / "resnet50_head_latest.pth"
)

LAYER4_BEST_PATH = (
    CHECKPOINT_DIR
    / "resnet50_layer4_best.pth"
)

LAYER4_LATEST_PATH = (
    CHECKPOINT_DIR
    / "resnet50_layer4_latest.pth"
)

print("Output directory:")
print(OUTPUT_DIR)

Output directory:
/kaggle/working/classifier_experiment


In [82]:
def set_batchnorm_eval(model):
    """
    Keep BatchNorm running statistics fixed during
    transfer learning with small batches.
    """

    for module in model.modules():

        if isinstance(
            module,
            nn.BatchNorm2d,
        ):
            module.eval()


def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device,
):

    model.train()

    set_batchnorm_eval(model)

    running_loss = 0.0

    for images, labels, _ in tqdm(
        loader,
        desc="Training",
        leave=False,
    ):

        images = images.to(
            device,
            non_blocking=True,
        )

        labels = (
            labels
            .float()
            .unsqueeze(1)
            .to(
                device,
                non_blocking=True,
            )
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(
                device.type == "cuda"
            ),
        ):

            logits = model(images)

            loss = criterion(
                logits,
                labels,
            )

        scaler.scale(
            loss
        ).backward()

        scaler.step(
            optimizer
        )

        scaler.update()

        running_loss += (
            loss.item()
        )

    return (
        running_loss / len(loader)
    )

In [83]:
def evaluate(
    model,
    loader,
    criterion,
    device,
    threshold=0.5,
):

    model.eval()

    running_loss = 0.0

    all_probs = []
    all_targets = []
    all_image_keys = []

    with torch.inference_mode():

        for (
            images,
            labels,
            image_keys,
        ) in tqdm(
            loader,
            desc="Validation",
            leave=False,
        ):

            images = images.to(
                device,
                non_blocking=True,
            )

            targets = (
                labels
                .float()
                .unsqueeze(1)
                .to(
                    device,
                    non_blocking=True,
                )
            )

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=(
                    device.type == "cuda"
                ),
            ):

                logits = model(images)

                loss = criterion(
                    logits,
                    targets,
                )

            probabilities = (
                torch.sigmoid(logits)
            )

            running_loss += (
                loss.item()
            )

            all_probs.extend(
                probabilities
                .squeeze(1)
                .float()
                .cpu()
                .numpy()
            )

            all_targets.extend(
                labels.numpy()
            )

            all_image_keys.extend(
                image_keys
            )

    probabilities = np.asarray(
        all_probs
    )

    targets = np.asarray(
        all_targets
    )

    predictions = (
        probabilities >= threshold
    ).astype(np.int64)

    tn, fp, fn, tp = (
        confusion_matrix(
            targets,
            predictions,
            labels=[0, 1],
        ).ravel()
    )

    sensitivity = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0.0
    )

    return {
        "loss": (
            running_loss
            / len(loader)
        ),
        "accuracy": accuracy_score(
            targets,
            predictions,
        ),
        "precision": precision_score(
            targets,
            predictions,
            zero_division=0,
        ),
        "sensitivity": sensitivity,
        "specificity": specificity,
        "f1": f1_score(
            targets,
            predictions,
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(
            targets,
            probabilities,
        ),
        "pr_auc": average_precision_score(
            targets,
            probabilities,
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "probabilities": probabilities,
        "targets": targets,
        "image_keys": all_image_keys,
    }

In [84]:
def save_checkpoint(
    path,
    epoch,
    model,
    optimizer,
    scaler,
    train_loss,
    val_metrics,
    stage,
):

    scalar_metrics = {
        key: value
        for key, value in val_metrics.items()
        if key not in {
            "probabilities",
            "targets",
            "image_keys",
        }
    }

    checkpoint = {
        "stage": stage,
        "epoch": epoch,
        "model_state_dict": (
            model.state_dict()
        ),
        "optimizer_state_dict": (
            optimizer.state_dict()
        ),
        "scaler_state_dict": (
            scaler.state_dict()
        ),
        "train_loss": (
            float(train_loss)
        ),
        "val_metrics": (
            scalar_metrics
        ),
        "configuration": {
            "target_height": TARGET_HEIGHT,
            "target_width": TARGET_WIDTH,
            "batch_size": BATCH_SIZE,
            "seed": SEED,
        },
    }

    torch.save(
        checkpoint,
        path,
    )

In [85]:
model = build_classifier(weights)
model = model.to(device)

criterion = (
    nn.BCEWithLogitsLoss()
)

optimizer = torch.optim.AdamW(
    model.fc.parameters(),
    lr=HEAD_LR,
    weight_decay=WEIGHT_DECAY,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(
        device.type == "cuda"
    ),
)

head_history = []

best_head_auc = -float("inf")

print("Official head-only training initialized.")

summarize_trainable_parameters(model)

Official head-only training initialized.
Total parameters: 23,510,081
Trainable parameters: 2,049

Trainable layers:
  fc.weight
  fc.bias


In [ ]:
for epoch in range(
    1,
    HEAD_EPOCHS + 1,
):

    print(
        f"\nEpoch {epoch}/{HEAD_EPOCHS}"
    )

    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        device=device,
    )

    val_metrics = evaluate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=device,
    )

    epoch_results = {
        "stage": "head",
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "accuracy": val_metrics["accuracy"],
        "precision": val_metrics["precision"],
        "sensitivity": val_metrics["sensitivity"],
        "specificity": val_metrics["specificity"],
        "f1": val_metrics["f1"],
        "roc_auc": val_metrics["roc_auc"],
        "pr_auc": val_metrics["pr_auc"],
    }

    head_history.append(
        epoch_results
    )

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f}"
    )

    print(
        f"Accuracy: {val_metrics['accuracy']:.4f} | "
        f"Sensitivity: {val_metrics['sensitivity']:.4f} | "
        f"Specificity: {val_metrics['specificity']:.4f}"
    )

    print(
        f"F1: {val_metrics['f1']:.4f} | "
        f"ROC-AUC: {val_metrics['roc_auc']:.4f} | "
        f"PR-AUC: {val_metrics['pr_auc']:.4f}"
    )

    save_checkpoint(
        path=HEAD_LATEST_PATH,
        epoch=epoch,
        model=model,
        optimizer=optimizer,
        scaler=scaler,
        train_loss=train_loss,
        val_metrics=val_metrics,
        stage="head",
    )

    if (
        val_metrics["roc_auc"]
        > best_head_auc
    ):

        best_head_auc = (
            val_metrics["roc_auc"]
        )

        save_checkpoint(
            path=HEAD_BEST_PATH,
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            scaler=scaler,
            train_loss=train_loss,
            val_metrics=val_metrics,
            stage="head",
        )

        print(
            "Saved new best head model "
            f"(ROC-AUC = "
            f"{best_head_auc:.4f})"
        )

In [ ]:
head_history_df = pd.DataFrame(
    head_history
)

head_history_df.to_csv(
    OUTPUT_DIR
    / "head_training_history.csv",
    index=False,
)

head_history_df

In [ ]:
assert HEAD_BEST_PATH.exists()

head_checkpoint = torch.load(
    HEAD_BEST_PATH,
    map_location=device,
    weights_only=False,
)

print(
    "Best head epoch:",
    head_checkpoint["epoch"],
)

print(
    "Best head ROC-AUC:",
    head_checkpoint[
        "val_metrics"
    ]["roc_auc"],
)

print("\nCheckpoint:")
print(HEAD_BEST_PATH)

In [ ]:
#Stage 2

In [ ]:
model = build_classifier(weights)
model = model.to(device)

checkpoint = torch.load(
    HEAD_BEST_PATH,
    map_location=device,
    weights_only=False,
)

model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)

print(
    "Loaded head checkpoint "
    f"from epoch "
    f"{checkpoint['epoch']}"
)

In [ ]:
def configure_layer4_finetuning(
    model
):

    # Freeze everything first
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze highest-level ResNet block
    for param in model.layer4.parameters():
        param.requires_grad = True

    # Classifier remains trainable
    for param in model.fc.parameters():
        param.requires_grad = True

    # Keep BatchNorm affine parameters frozen
    for module in model.modules():

        if isinstance(
            module,
            nn.BatchNorm2d,
        ):

            for param in module.parameters():
                param.requires_grad = False


configure_layer4_finetuning(model)

summarize_trainable_parameters(model)

In [ ]:
criterion = (
    nn.BCEWithLogitsLoss()
)

layer4_parameters = [
    param
    for param in model.layer4.parameters()
    if param.requires_grad
]

fc_parameters = [
    param
    for param in model.fc.parameters()
    if param.requires_grad
]

optimizer = torch.optim.AdamW(
    [
        {
            "params": layer4_parameters,
            "lr": LAYER4_LR,
        },
        {
            "params": fc_parameters,
            "lr": FC_FINETUNE_LR,
        },
    ],
    weight_decay=WEIGHT_DECAY,
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(
        device.type == "cuda"
    ),
)

print("Layer4 LR:", LAYER4_LR)
print("FC LR:", FC_FINETUNE_LR)

In [ ]:
finetune_history = []

best_finetune_auc = -float("inf")

epochs_without_improvement = 0


for epoch in range(
    1,
    FINETUNE_EPOCHS + 1,
):

    print(
        f"\nFine-tune Epoch "
        f"{epoch}/{FINETUNE_EPOCHS}"
    )

    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        device=device,
    )

    val_metrics = evaluate(
        model=model,
        loader=val_loader,
        criterion=criterion,
        device=device,
    )

    epoch_results = {
        "stage": "layer4",
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "accuracy": val_metrics["accuracy"],
        "precision": val_metrics["precision"],
        "sensitivity": val_metrics["sensitivity"],
        "specificity": val_metrics["specificity"],
        "f1": val_metrics["f1"],
        "roc_auc": val_metrics["roc_auc"],
        "pr_auc": val_metrics["pr_auc"],
    }

    finetune_history.append(
        epoch_results
    )

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f}"
    )

    print(
        f"Accuracy: {val_metrics['accuracy']:.4f} | "
        f"Sensitivity: {val_metrics['sensitivity']:.4f} | "
        f"Specificity: {val_metrics['specificity']:.4f}"
    )

    print(
        f"F1: {val_metrics['f1']:.4f} | "
        f"ROC-AUC: {val_metrics['roc_auc']:.4f} | "
        f"PR-AUC: {val_metrics['pr_auc']:.4f}"
    )

    save_checkpoint(
        path=LAYER4_LATEST_PATH,
        epoch=epoch,
        model=model,
        optimizer=optimizer,
        scaler=scaler,
        train_loss=train_loss,
        val_metrics=val_metrics,
        stage="layer4",
    )

    if (
        val_metrics["roc_auc"]
        > best_finetune_auc
    ):

        best_finetune_auc = (
            val_metrics["roc_auc"]
        )

        epochs_without_improvement = 0

        save_checkpoint(
            path=LAYER4_BEST_PATH,
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            scaler=scaler,
            train_loss=train_loss,
            val_metrics=val_metrics,
            stage="layer4",
        )

        print(
            "Saved new best fine-tuned model "
            f"(ROC-AUC = "
            f"{best_finetune_auc:.4f})"
        )

    else:

        epochs_without_improvement += 1

        print(
            "Epochs without ROC-AUC "
            "improvement:",
            epochs_without_improvement,
        )

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):

        print(
            "\nEarly stopping triggered."
        )

        break

In [ ]:
finetune_history_df = pd.DataFrame(
    finetune_history
)

finetune_history_df.to_csv(
    OUTPUT_DIR
    / "layer4_training_history.csv",
    index=False,
)

finetune_history_df

In [ ]:
head_checkpoint = torch.load(
    HEAD_BEST_PATH,
    map_location="cpu",
    weights_only=False,
)

layer4_checkpoint = torch.load(
    LAYER4_BEST_PATH,
    map_location="cpu",
    weights_only=False,
)

head_auc = (
    head_checkpoint[
        "val_metrics"
    ]["roc_auc"]
)

layer4_auc = (
    layer4_checkpoint[
        "val_metrics"
    ]["roc_auc"]
)

print(
    f"Best head ROC-AUC:   "
    f"{head_auc:.4f}"
)

print(
    f"Best layer4 ROC-AUC: "
    f"{layer4_auc:.4f}"
)


if layer4_auc > head_auc:

    SELECTED_CHECKPOINT = (
        LAYER4_BEST_PATH
    )

    selected_stage = "layer4"

else:

    SELECTED_CHECKPOINT = (
        HEAD_BEST_PATH
    )

    selected_stage = "head"


print("\nSelected model:")
print(selected_stage)

print(
    SELECTED_CHECKPOINT
)

In [ ]:
training_summary = {
    "selected_stage": selected_stage,
    "selected_checkpoint": (
        SELECTED_CHECKPOINT.name
    ),
    "best_head_roc_auc": float(
        head_auc
    ),
    "best_layer4_roc_auc": float(
        layer4_auc
    ),
    "target_resolution": [
        TARGET_HEIGHT,
        TARGET_WIDTH,
    ],
    "batch_size": BATCH_SIZE,
    "seed": SEED,
}

with open(
    OUTPUT_DIR
    / "training_summary.json",
    "w",
) as f:

    json.dump(
        training_summary,
        f,
        indent=2,
    )


print(
    json.dumps(
        training_summary,
        indent=2,
    )
)